# Convert EPLAN Assistant v2 to GGUF

Downloads merged model from HuggingFace, converts to GGUF, quantizes to Q4_K_M, and uploads back.

In [ ]:
# Step 1: Download merged model from HuggingFace
from huggingface_hub import snapshot_download, login
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF_TOKEN"))
print("Logged in!")

print("Downloading merged model (~6GB)...")
snapshot_download("covaga/eplan-assistant-v2-merged", local_dir="eplan-assistant-v2-merged")
print("Done!")

In [ ]:
import subprocess, os

# Step 2: Install build tools
print("Installing build tools...")
subprocess.run(["apt-get", "update", "-qq"], check=True, capture_output=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "build-essential", "cmake"], check=True, capture_output=True)

# Step 3: Clone llama.cpp
print("Cloning llama.cpp...")
if not os.path.exists("/tmp/llama.cpp"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggerganov/llama.cpp.git", "/tmp/llama.cpp"], check=True)

# Step 4: Install Python deps
print("Installing Python deps...")
subprocess.run(["pip", "install", "-q", "gguf", "sentencepiece", "protobuf"], check=True, capture_output=True)

# Step 5: Convert to FP16 GGUF
print("Converting to GGUF FP16...")
subprocess.run([
    "python", "/tmp/llama.cpp/convert_hf_to_gguf.py",
    "eplan-assistant-v2-merged",
    "--outfile", "eplan-assistant-v2-f16.gguf",
    "--outtype", "f16",
], check=True)
print("FP16 GGUF created!")

# Step 6: Build quantize tool
print("Building quantize tool...")
os.makedirs("/tmp/llama.cpp/build", exist_ok=True)
subprocess.run([
    "cmake", "-B", "/tmp/llama.cpp/build", "-S", "/tmp/llama.cpp", "-DGGML_CUDA=OFF"
], check=True, capture_output=True)
subprocess.run([
    "cmake", "--build", "/tmp/llama.cpp/build", "--target", "llama-quantize", "-j", "4"
], check=True, capture_output=True)
print("Quantize tool built!")

# Step 7: Quantize
quantize_bin = "/tmp/llama.cpp/build/bin/llama-quantize"
print("Quantizing to Q4_K_M...")
subprocess.run([
    quantize_bin,
    "eplan-assistant-v2-f16.gguf",
    "eplan-assistant-v2-q4_k_m.gguf",
    "Q4_K_M",
], check=True)

f16_size = os.path.getsize("eplan-assistant-v2-f16.gguf") / (1024**3)
q4_size = os.path.getsize("eplan-assistant-v2-q4_k_m.gguf") / (1024**3)
print(f"\nF16:    {f16_size:.2f} GB")
print(f"Q4_K_M: {q4_size:.2f} GB")
print(f"Compression: {f16_size/q4_size:.1f}x")

In [ ]:
# Re-login (Kaggle doesn't persist auth between cells)
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

api = HfApi()
repo_id = "covaga/eplan-assistant-v2-gguf"
api.create_repo(repo_id, exist_ok=True)

print("Uploading Q4_K_M GGUF...")
api.upload_file(
    path_or_fileobj="eplan-assistant-v2-q4_k_m.gguf",
    path_in_repo="eplan-assistant-v2-q4_k_m.gguf",
    repo_id=repo_id,
    commit_message="Add Q4_K_M GGUF (quantized from v2 merged model)",
)
print(f"Done! Model at: https://huggingface.co/{repo_id}")
print(f"\nTo use with Ollama:")
print(f"  huggingface-cli download {repo_id} eplan-assistant-v2-q4_k_m.gguf --local-dir .")
print(f'  echo "FROM ./eplan-assistant-v2-q4_k_m.gguf" > Modelfile')
print(f"  ollama create eplan-assistant -f Modelfile")
print(f"  ollama run eplan-assistant")